# Model testing

In [1]:
from fire_spread import fire_spread_model, clip_study_area, data_preparation_functions
import geopandas as gpd
import rasterio

In [2]:
# Load full data
fires_gpd = gpd.read_file("../01_Data/06_Wildfire_clusters/Fire_clusters_chaco.shp")

In [25]:
event_id = 3652  # Example event ID
year = fires_gpd[fires_gpd['CLUSTER_ID'] == event_id]['ACQ_DATE'].iloc[0].year

In [26]:
# Get earliest ignition point for the event
ignition_point = fires_gpd[fires_gpd['CLUSTER_ID'] == event_id].sort_values('ACQ_DATE')

In [27]:
# Get ERA5 data for the specific event
era5_data = data_preparation_functions.get_era5_data(
    gdf_init = ignition_point, 
    id_column = 'CLUSTER_ID', 
    lat = None, 
    lon = None, 
    start_date = 'START_TIME', 
    end_date = 'END_TIME', 
    delta_end = 5, 
    delta_start = 5, 
    ee_project = 'webprogrammingumd', 
    event_id = event_id
    )

Taking the first ignition point as representative.
Extracted 240 hourly records for event 3652


In [28]:
path = f"../01_Data/05_Spread_Covariates/ERA5_event_{event_id}.csv"

In [29]:
era5_data[event_id].to_csv(path, index=False)

In [36]:
from fire_spread import monte_carlo as mc
from fire_spread import data_preparation_functions as data_prep
from fire_spread import fire_spread_model as fire_model

Kr = 5
R0 = 0.8
delta_t = None
buffer = 30
m = 0.8

In [34]:
# 1. Prepare data
ca_data = data_prep.prepare_ca_inputs(
    event_id=event_id, 
    land_use_path=f'../01_Data/03_MapBiomas/{year}_coverage_lclu_25-1-1.tif',
    weather_csv_path= path,
    srtm_path= '../01_Data/07_SRTM/SRTM_Paraguay_Chaco.tif',
    fire_points_gdf=fires_gpd, buffer_km=buffer)


Preparing CA inputs for event 3652

1. Loading elevation...
  Elevation loaded: (22256, 20369)
  Valid cells: 453332464
  NoData cells: 0
  Elevation range: 0.0 to 623.0 m

2. Finding ignition location...
  Ignition point (full grid): row=19040, col=16952
  Coordinates: x=-58.076500, y=-24.418400
  Ignition time: 2021-08-19 00:00:00

3. Clipping to 30 km buffer around ignition...
  Original grid: 22,256 × 20,369 = 453,332,464 cells (453.3 million)
  Clipped grid:  2,001 × 2,001 = 4,004,001 cells (4.00 million)
  Buffer: 30 km (1000 cells)
  Memory reduction: 99.1%
  Ignition point (clipped grid): row=1000, col=1000

4. Calculating slope and aspect...
  Slope calculated: range 0.0 to 14.2 degrees
  Aspect calculated: range 0 to 6.18 radians

5. Loading land use...
  Land use loaded: (30849, 31147)
  Unique classes: 11

6. Clipping land use to same extent...
  Land use needs snapping before clipping...

7. Mapping to fuel types...
  Fuel type distribution:
    Type 1 (Ks=0.40): 670,602 

In [37]:

# 2. Run Monte Carlo (50 realizations)

mc_results = mc.run_monte_carlo_simulation(
    ca_data=ca_data,
    ca_model=fire_model,
    n_runs=50,
    Kr=Kr,
    R0 = R0,
    delta_t = delta_t,
    m = m,
    seed=42  # Reproducibility
)

# 3. Export results
mc.export_monte_carlo_results(
    mc_results,
    output_dir=f'results/event_{event_id}_{Kr}_{R0}_{delta_t}_{buffer}',
    event_id=event_id,
    Kr=Kr
)


Running Monte Carlo Simulation: 50 realizations
Parameters: Kr=5, max_time_steps=1000

Running simulations...


Progress:   0%|          | 0/50 [00:00<?, ?it/s]

  Adaptive time step calculation:
    Wind (90th percentile): 6.69 m/s
    Wind factor: 1.351
    Slope factor: 1.042
    Rmax: 5.630 m/min
    Delta_t: 4.263 minutes (255.8 seconds)
t=0: Burning cells: 1, Time: 2021-08-19 00:00:00
  → New ignitions this timestep: 3
t=1: Burning cells: 3, Time: 2021-08-19 00:04:15.776388422
  → New ignitions this timestep: 8
t=2: Burning cells: 8, Time: 2021-08-19 00:08:31.552776844
  → New ignitions this timestep: 9
t=3: Burning cells: 8, Time: 2021-08-19 00:12:47.329165267
  → New ignitions this timestep: 10
t=4: Burning cells: 8, Time: 2021-08-19 00:17:03.105553689
  → New ignitions this timestep: 9
t=5: Burning cells: 7, Time: 2021-08-19 00:21:18.881942112
  → New ignitions this timestep: 10
t=6: Burning cells: 8, Time: 2021-08-19 00:25:34.658330534
  → New ignitions this timestep: 14
t=7: Burning cells: 12, Time: 2021-08-19 00:29:50.434718957
  → New ignitions this timestep: 11
t=8: Burning cells: 10, Time: 2021-08-19 00:34:06.211107379
  → New ig

Progress:   2%|▏         | 1/50 [00:27<22:17, 27.30s/it]

t=999: Burning cells: 739, Time: 2021-08-21 22:58:40.612034064
  Run 1: 311209 cells burned
  Adaptive time step calculation:
    Wind (90th percentile): 6.69 m/s
    Wind factor: 1.351
    Slope factor: 1.042
    Rmax: 5.630 m/min
    Delta_t: 4.263 minutes (255.8 seconds)
t=0: Burning cells: 1, Time: 2021-08-19 00:00:00
  → New ignitions this timestep: 6
t=1: Burning cells: 6, Time: 2021-08-19 00:04:15.776388422
  → New ignitions this timestep: 3
t=2: Burning cells: 3, Time: 2021-08-19 00:08:31.552776844
  → New ignitions this timestep: 5
t=3: Burning cells: 5, Time: 2021-08-19 00:12:47.329165267
  → New ignitions this timestep: 9
t=4: Burning cells: 8, Time: 2021-08-19 00:17:03.105553689
  → New ignitions this timestep: 8
t=5: Burning cells: 7, Time: 2021-08-19 00:21:18.881942112
  → New ignitions this timestep: 12
t=6: Burning cells: 11, Time: 2021-08-19 00:25:34.658330534
  → New ignitions this timestep: 10
t=7: Burning cells: 9, Time: 2021-08-19 00:29:50.434718957
  → New ignitio

...

  Adaptive time step calculation:
    Wind (90th percentile): 6.69 m/s
    Wind factor: 1.351
    Slope factor: 1.042
    Rmax: 5.630 m/min
    Delta_t: 4.263 minutes (255.8 seconds)
t=0: Burning cells: 1, Time: 2021-08-19 00:00:00
  → New ignitions this timestep: 2
t=1: Burning cells: 2, Time: 2021-08-19 00:04:15.776388422
  → New ignitions this timestep: 1
t=2: Burning cells: 1, Time: 2021-08-19 00:08:31.552776844
  → New ignitions this timestep: 1
t=3: Burning cells: 1, Time: 2021-08-19 00:12:47.329165267
  → New ignitions this timestep: 3
t=4: Burning cells: 3, Time: 2021-08-19 00:17:03.105553689
  → New ignitions this timestep: 4
t=5: Burning cells: 4, Time: 2021-08-19 00:21:18.881942112
  → New ignitions this timestep: 6
t=6: Burning cells: 5, Time: 2021-08-19 00:25:34.658330534
  → New ignitions this timestep: 5
t=7: Burning cells: 4, Time: 2021-08-19 00:29:50.434718957
  → New ignitions this timestep: 4
t=8: Burning cells: 4, Time: 2021-08-19 00:34:06.211107379
  → New ignition

Progress: 100%|██████████| 50/50 [24:33<00:00, 29.48s/it]

  Run 50: 321747 cells burned

Aggregating results...




c:\Users\pmedi\Documents\09_Research\Wildfire Prediction\05_Notebooks\02_Fire_Spread\fire_spread\monte_carlo.py:140: RuntimeWarning: invalid value encountered in divide
  burn_time_sum / burn_count,
c:\Users\pmedi\Documents\09_Research\Wildfire Prediction\05_Notebooks\02_Fire_Spread\fire_spread\monte_carlo.py:148: RuntimeWarning: invalid value encountered in divide
  (burn_time_squared_sum / burn_count) - (burn_time_sum / burn_count) ** 2,


Monte Carlo Simulation Summary:
  Number of runs: 50
  Kr parameter: 5

Burned Cells Statistics:
  Mean:   280927.1 cells
  Median: 321575 cells
  Std:    126633.9 cells
  Min:    1 cells
  Max:    430463 cells
  Q25:    301903 cells
  Q75:    348302 cells

Simulation Termination:
  Extinguished early: 16.0%
  Reached max steps:  84.0%

Burned Area Statistics (hectares):
  Mean:   25283.44 ha
  Median: 28941.75 ha
  Std:    11397.05 ha



Exporting Monte Carlo results to results/event_3652_5_0.8_None_30/...
  Exported: results/event_3652_5_0.8_None_30/event_3652_Kr_5_burn_probability.tif
  Exported: results/event_3652_5_0.8_None_30/event_3652_Kr_5_mean_burn_time.tif
  Exported: results/event_3652_5_0.8_None_30/event_3652_Kr_5_std_burn_time.tif
  Exported: results/event_3652_5_0.8_None_30/event_3652_Kr_5_ci_lower_95.tif
  Exported: results/event_3652_5_0.8_None_30/event_3652_Kr_5_ci_upper_95.tif
  Exported: results/event_3652_5_0.8_None_30/event_3652_Kr_5_burn_count.tif
  Exported: resu